In [ ]:
!pip install ultralytics deep-sort-realtime opencv-python-headless matplotlib numpy pandas fastapi uvicorn python-multipart --quiet

In [ ]:
import sys
import torch

print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Select device: 'cuda' if GPU is present, otherwise 'cpu'
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

if DEVICE == "cuda":
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

In [ ]:
import urllib.request
import os

VIDEO_URL = "https://commondatastorage.googleapis.com/gtv-videos-bucket/sample/ForBiggerBlazes.mp4"
VIDEO_PATH = "sample_video.mp4"

if not os.path.exists(VIDEO_PATH):
    print("Downloading sample video...")
    urllib.request.urlretrieve(VIDEO_URL, VIDEO_PATH)
    print("Download complete.")
else:
    print("Video already exists, skipping download.")

print(f"Video path: {VIDEO_PATH}")

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 nano model
model = YOLO("yolov8n.pt")

# Move to correct device
model.to(DEVICE)

print("YOLOv8 model loaded successfully.")
print(f"Model is running on: {next(model.model.parameters()).device}")

In [ ]:
import cv2
import matplotlib.pyplot as plt

def load_first_frame(video_path: str):
    """Open a video file and return the first frame as a NumPy array (BGR)."""
    cap = cv2.VideoCapture(video_path)
    success, frame = cap.read()
    cap.release()

    if not success:
        raise RuntimeError(f"Could not read frame from: {video_path}")

    return frame


def show_frame(frame, title="Frame"):
    """Display an OpenCV BGR frame inline in Jupyter."""
    # Convert BGR → RGB for matplotlib
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 7))
    plt.imshow(rgb)
    plt.title(title, fontsize=14)
    plt.axis("off")
    plt.tight_layout()
    plt.show()


# Test it
frame = load_first_frame(VIDEO_PATH)
print(f"Frame shape: {frame.shape}  (height x width x channels)")
show_frame(frame, title="Raw Input Frame")

In [ ]:
CONFIDENCE_THRESHOLD = 0.4

# Run inference
results = model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)

# results[0] holds detections for our single frame
result = results[0]

print(f"Number of detections: {len(result.boxes)}")
print(f"Class names available: {list(result.names.values())[:10]}")

In [ ]:
annotated_frame = result.plot()
show_frame(annotated_frame, title="YOLOv8 Detections (raw output)")

In [ ]:
for i, box in enumerate(result.boxes[:3]): # Show first 3 detections
    x1, y1, x2, y2 = box.xyxy[0].tolist() # Bounding box corners
    conf = box.conf[0].item() # Confidence score
    cls_id = int(box.cls[0].item()) # Class index
    cls_name = result.names[cls_id] # Class name string

    print(f"Detection {i+1}:")
    print(f"BBox (x1,y1,x2,y2): ({x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f})")
    print(f"Confidence: {conf:.3f}")
    print(f"Class: {cls_name} (id={cls_id})")
    print()

In [ ]:
from typing import List, Tuple
import numpy as np


def extract_detections(
    result,
    target_classes: List[str] = None
) -> List[Tuple[List[float], float, str]]:

    detections = []

    for box in result.boxes:
        # Extract values from tensor → Python floats
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        conf = float(box.conf[0])
        cls_id = int(box.cls[0])
        cls_name = result.names[cls_id]

        # Filter by class if requested
        if target_classes and cls_name not in target_classes:
            continue

        # Convert from [x1, y1, x2, y2] → [x1, y1, width, height]
        width = x2 - x1
        height = y2 - y1
        bbox = [x1, y1, width, height]

        detections.append((bbox, conf, cls_name))

    return detections


# Test it on our frame
detections = extract_detections(result)
print(f"Total detections extracted: {len(detections)}")
print("\nFirst 3 formatted detections:")
for det in detections[:3]:
    bbox, conf, cls_name = det
    print(f"  class={cls_name:10s}  conf={conf:.3f}  bbox=[x={bbox[0]:.0f}, y={bbox[1]:.0f}, w={bbox[2]:.0f}, h={bbox[3]:.0f}]")

In [ ]:
from deep_sort_realtime.deepsort_tracker import DeepSort

tracker = DeepSort(
    max_age=30, # A track lives 30 frames without a detection
    n_init=3, # Need 3 consecutive detections to confirm a track
    max_iou_distance=0.7, # Max overlap distance for matching
    embedder="mobilenet", # Appearance feature extractor (for re-ID)
    half=True, # Use FP16 if on GPU for speed
    bgr=True, # Our frames are BGR (OpenCV default)
)

print("Deep SORT tracker initialized.")

In [ ]:
def update_tracker(
    tracker: DeepSort,
    frame: np.ndarray,
    detections: List[Tuple]
) -> List[dict]:
    # Update tracker — this runs Kalman prediction + Hungarian matching
    raw_tracks = tracker.update_tracks(detections, frame=frame)

    confirmed_tracks = []

    for track in raw_tracks:
        # Skip tentative (unconfirmed) tracks
        # A track needs n_init=3 frames before we trust it
        if not track.is_confirmed():
            continue

        track_id = track.track_id
        bbox = track.to_ltrb()  # [left, top, right, bottom] = [x1, y1, x2, y2]
        cls_name = track.get_det_class() or "unknown"

        confirmed_tracks.append({
            "track_id": track_id,
            "bbox": [int(v) for v in bbox],  # Convert to int for drawing
            "class_name": cls_name
        })

    return confirmed_tracks


print("Tracking function defined.")

In [ ]:
from collections import defaultdict
import pandas as pd


class TrajectoryStore:

    def __init__(self):
        # defaultdict(list) creates [] automatically for new keys
        self.data = defaultdict(list)

    def update(self, frame_number: int, tracks: List[dict]):
        """
        Record current positions of all confirmed tracks.

        Args:
            frame_number: Current frame index in the video
            tracks: Output of update_tracker()
        """
        for track in tracks:
            x1, y1, x2, y2 = track["bbox"]

            # Compute bounding box center
            cx = (x1 + x2) / 2.0
            cy = (y1 + y2) / 2.0

            self.data[track["track_id"]].append({
                "frame":      frame_number,
                "cx":         cx,
                "cy":         cy,
                "class_name": track["class_name"]
            })

    def get_trajectory(self, track_id: int) -> List[dict]:
        """Return position history for a specific track."""
        return self.data.get(track_id, [])

    def get_all_ids(self) -> List[int]:
        """Return all known track IDs."""
        return list(self.data.keys())

    def to_dataframe(self) -> pd.DataFrame:
        """
        Convert all trajectories into a flat DataFrame.
        Useful for analysis and export.

        Columns: track_id, frame, cx, cy, class_name
        """
        rows = []
        for track_id, points in self.data.items():
            for point in points:
                rows.append({"track_id": track_id, **point})
        return pd.DataFrame(rows)

    def clear(self):
        """Reset all stored trajectories."""
        self.data.clear()


# Initialize the store
trajectory_store = TrajectoryStore()
print("TrajectoryStore initialized.")

In [ ]:
MAX_FRAMES = 150  # Adjust up for longer analysis

cap = cv2.VideoCapture(VIDEO_PATH)
tracker.delete_all_tracks()
trajectory_store.clear()

frame_number = 0

while frame_number < MAX_FRAMES:
    success, frame = cap.read()
    if not success:
        break  # End of video

    # Detect
    results = model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)
    detections = extract_detections(results[0])

    # Track
    tracks = update_tracker(tracker, frame, detections)

    # Store trajectory
    trajectory_store.update(frame_number, tracks)

    frame_number += 1

cap.release()

print(f"Processed {frame_number} frames.")
print(f"Unique tracked objects: {len(trajectory_store.get_all_ids())}")

In [ ]:
df = trajectory_store.to_dataframe()
print("Trajectory DataFrame shape:", df.shape)
print("\nFirst 10 rows:")
df.head(10)

In [ ]:
summary = df.groupby("track_id").agg(
    class_name=("class_name", "first"),
    frames_seen=("frame", "count"),
    first_frame=("frame", "min"),
    last_frame=("frame", "max"),
).reset_index()

print("Per-track summary:")
print(summary.to_string(index=False))

In [ ]:
def compute_motion_features(trajectory: List[dict]) -> dict:
    """
    Compute motion statistics for a single track's trajectory.

    Args:
        trajectory: List of {frame, cx, cy, class_name} dicts

    Returns:
        dict with motion statistics
    """
    if len(trajectory) < 2:
        return {"speeds": [], "directions": [], "path_length": 0, "displacement": 0}

    speeds     = []
    directions = []
    path_length = 0.0

    for i in range(1, len(trajectory)):
        prev = trajectory[i - 1]
        curr = trajectory[i]

        dx = curr["cx"] - prev["cx"]
        dy = curr["cy"] - prev["cy"]
        dt = max(curr["frame"] - prev["frame"], 1)  # Avoid divide-by-zero

        # Euclidean distance between consecutive positions
        dist = np.sqrt(dx**2 + dy**2)

        # Speed = distance / time (pixels per frame)
        speed = dist / dt
        speeds.append(speed)
        path_length += dist

        # Direction = angle of motion vector (degrees, 0=right, 90=down)
        angle = np.degrees(np.arctan2(dy, dx))
        directions.append(angle)

    # Net displacement = straight-line distance from start to end
    start = trajectory[0]
    end   = trajectory[-1]
    displacement = np.sqrt(
        (end["cx"] - start["cx"])**2 + (end["cy"] - start["cy"])**2
    )

    return {
        "speeds":      speeds,
        "directions":  directions,
        "path_length": path_length,
        "displacement": displacement
    }


# Test on one track
sample_id = trajectory_store.get_all_ids()[0]
sample_traj = trajectory_store.get_trajectory(sample_id)
features = compute_motion_features(sample_traj)

print(f"Track ID: {sample_id}")
print(f"  Points in trajectory : {len(sample_traj)}")
print(f"  Path length (px)     : {features['path_length']:.1f}")
print(f"  Net displacement (px): {features['displacement']:.1f}")
if features['speeds']:
    print(f"  Avg speed (px/frame) : {np.mean(features['speeds']):.2f}")
    print(f"  Max speed (px/frame) : {np.max(features['speeds']):.2f}")

In [ ]:
SPEED_FAST_THRESHOLD = 8.0 # pixels/frame
SPEED_STOP_THRESHOLD = 2.0 # pixels/frame
DIRECTION_CHANGE_DEGREES = 60.0 # degrees

def detect_behaviors(features: dict) -> dict:
    """
    Flag behavioral events given motion features.

    Returns:
        dict with boolean behavior flags
    """
    speeds     = features["speeds"]
    directions = features["directions"]

    behaviors = {
        "is_fast_moving":      False,
        "is_suddenly_stopped": False,
        "has_direction_change": False
    }

    if not speeds:
        return behaviors

    # 1. Fast movement: any frame with speed above threshold
    behaviors["is_fast_moving"] = any(s > SPEED_FAST_THRESHOLD for s in speeds)

    # 2. Sudden stop: was fast (above threshold) → then slow (below stop threshold)
    for i in range(1, len(speeds)):
        if speeds[i - 1] > SPEED_FAST_THRESHOLD and speeds[i] < SPEED_STOP_THRESHOLD:
            behaviors["is_suddenly_stopped"] = True
            break

    # 3. Direction change: angle difference between consecutive steps > threshold
    for i in range(1, len(directions)):
        delta = abs(directions[i] - directions[i - 1])
        # Normalize: account for angle wrapping (e.g., 170° → -170°)
        delta = min(delta, 360 - delta)
        if delta > DIRECTION_CHANGE_DEGREES:
            behaviors["has_direction_change"] = True
            break

    return behaviors


# Test
behaviors = detect_behaviors(features)
print(f"Behaviors for track {sample_id}: {behaviors}")

In [ ]:
behavior_report = []

for track_id in trajectory_store.get_all_ids():
    traj = trajectory_store.get_trajectory(track_id)
    features = compute_motion_features(traj)
    behaviors = detect_behaviors(features)

    behavior_report.append({
        "track_id":    track_id,
        "class_name":  traj[0]["class_name"] if traj else "unknown",
        "frames_seen": len(traj),
        "path_length": round(features["path_length"], 1),
        "displacement": round(features["displacement"], 1),
        "avg_speed":   round(np.mean(features["speeds"]), 2) if features["speeds"] else 0,
        **behaviors
    })

behavior_df = pd.DataFrame(behavior_report)
print("Behavior Analysis Report:")
behavior_df

In [ ]:
from typing import Tuple, Union

def get_track_color(track_id: Union[int, str]) -> Tuple[int, int, int]:
    palette = [
        (255,  80,  80),
        ( 80, 255,  80),
        ( 80,  80, 255),
        (255, 255,  80),
        (255,  80, 255),
        ( 80, 255, 255),
        (255, 165,   0),
        (128,   0, 255),
        (  0, 200, 200),
        (200, 200,   0),
    ]
    # Convert track_id to an integer before using it for the modulo operation
    numeric_track_id = int(track_id)
    return palette[numeric_track_id % len(palette)]


print("Color palette ready.")

In [ ]:
TRAJECTORY_MAX_LEN = 40 # Draw only the last N positions


def annotate_frame(
    frame: np.ndarray,
    tracks: List[dict],
    trajectory_store: TrajectoryStore,
    behavior_flags: dict = None
) -> np.ndarray:
    """
    Annotate a video frame with:
      - Bounding boxes
      - Track ID labels
      - Trajectory paths
      - Behavior alerts

    Args:
        frame            : BGR frame to annotate (will be modified in-place copy)
        tracks           : Current frame's confirmed tracks
        trajectory_store : History of all track positions
        behavior_flags   : Dict[track_id → behaviors dict]

    Returns:
        Annotated BGR frame (NumPy array)
    """
    vis = frame.copy()
    behavior_flags = behavior_flags or {}

    # Draw trajectories (behind bounding boxes)
    for track_id, points in trajectory_store.data.items():
        color = get_track_color(track_id)

        # Only draw the last TRAJECTORY_MAX_LEN points
        recent = points[-TRAJECTORY_MAX_LEN:]

        for i in range(1, len(recent)):
            pt1 = (int(recent[i-1]["cx"]), int(recent[i-1]["cy"]))
            pt2 = (int(recent[i]["cx"]),   int(recent[i]["cy"]))

            # Fade: older segments are thinner and dimmer
            alpha = i / len(recent)        # 0 → 1 (older → newer)
            fade_color = tuple(int(c * alpha) for c in color)
            thickness = max(1, int(3 * alpha))

            cv2.line(vis, pt1, pt2, fade_color, thickness, lineType=cv2.LINE_AA)

    # ── Draw bounding boxes and labels
    for track in tracks:
        tid = track["track_id"]
        x1, y1, x2, y2 = track["bbox"]
        cls_name = track["class_name"]
        color = get_track_color(tid)

        # Bounding box rectangle
        cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)

        # Label: "ID:3 person"
        label = f"ID:{tid} {cls_name}"

        # Check for behavior flags
        flags = behavior_flags.get(tid, {})
        alerts = []
        if flags.get("is_fast_moving"): alerts.append("FAST")
        if flags.get("is_suddenly_stopped"): alerts.append("STOP")
        if flags.get("has_direction_change"): alerts.append("TURN")

        # Draw main label
        cv2.putText(
            vis, label,
            (x1, max(y1 - 8, 12)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA
        )

        # Draw behavior alerts in red
        if alerts:
            alert_str = " | ".join(alerts)
            cv2.putText(
                vis, alert_str,
                (x1, max(y1 - 26, 26)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 255), 2, cv2.LINE_AA
            )

    return vis


print("annotate_frame() defined.")

In [ ]:
OUTPUT_VIDEO_PATH = "output_tracked.mp4"
MAX_FRAMES_OUTPUT = 200  # Increase for longer video

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# mp4v codec — broadly compatible
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (W, H))

# Reset all state
tracker.delete_all_tracks()
trajectory_store.clear()

frame_number = 0

while frame_number < MAX_FRAMES_OUTPUT:
    success, frame = cap.read()
    if not success:
        break

    # ── 1. Detect
    results = model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)
    detections = extract_detections(results[0])

    # ── 2. Track
    tracks = update_tracker(tracker, frame, detections)

    # ── 3. Store
    trajectory_store.update(frame_number, tracks)

    # ── 4. Analyze behavior (live, per-frame)
    live_behaviors = {}
    for track in tracks:
        tid = track["track_id"]
        traj = trajectory_store.get_trajectory(tid)
        feats = compute_motion_features(traj)
        live_behaviors[tid] = detect_behaviors(feats)

    # ── 5. Annotate
    annotated = annotate_frame(frame, tracks, trajectory_store, live_behaviors)

    # ── 6. Write to disk
    writer.write(annotated)

    frame_number += 1
    if frame_number % 30 == 0:
        print(f"  Processed frame {frame_number}/{MAX_FRAMES_OUTPUT}")

cap.release()
writer.release()
print(f"\nOutput video saved to: {OUTPUT_VIDEO_PATH}")

In [ ]:
def show_video_frames(video_path: str, num_frames: int = 3):
    """Display evenly-spaced frames from a video inline."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = [int(total * i / num_frames) for i in range(num_frames)]

    fig, axes = plt.subplots(1, num_frames, figsize=(18, 5))

    for ax, idx in zip(axes, indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        success, frame = cap.read()
        if success:
            ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            ax.set_title(f"Frame {idx}", fontsize=11)
        ax.axis("off")

    cap.release()
    plt.suptitle("Annotated Output Frames", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


show_video_frames(OUTPUT_VIDEO_PATH, num_frames=4)
print("New video path: ", OUTPUT_VIDEO_PATH)

In [ ]:
def compute_tracking_metrics(trajectory_store: TrajectoryStore) -> dict:
    lengths = [len(pts) for pts in trajectory_store.data.values()]

    if not lengths:
        return {}

    short_tracks = sum(1 for l in lengths if l < 5)

    return {
        "total_tracks":      len(lengths),
        "avg_track_length":  round(np.mean(lengths), 1),
        "median_track_length": int(np.median(lengths)),
        "max_track_length":  max(lengths),
        "short_tracks_pct":  round(short_tracks / len(lengths) * 100, 1)
    }


metrics = compute_tracking_metrics(trajectory_store)

print("=== Tracking Quality Report ===")
for key, val in metrics.items():
    print(f"  {key:<25}: {val}")

print("\nInterpretation:")
print(f"  avg_track_length > 20 frames  → stable tracking")
print(f"  short_tracks_pct < 30%        → low fragmentation")

In [ ]:
lengths = [len(pts) for pts in trajectory_store.data.values()]

plt.figure(figsize=(9, 4))
plt.hist(lengths, bins=20, color="steelblue", edgecolor="white")
plt.axvline(np.mean(lengths), color="orange", linestyle="--", label=f"Mean = {np.mean(lengths):.1f}")
plt.xlabel("Track length (frames)")
plt.ylabel("Count")
plt.title("Distribution of Track Lengths")
plt.legend()
plt.tight_layout()
plt.show()